In [ ]:
pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLO26 nano model (best for fast, real-time webcam processing)
model = YOLO('yolo26n.pt')

# Source '0' opens the default computer webcam. 

# 'show=True' pops up a window for the class to see the results live.
# 'save=True' saves the resulting video to your computer.
results = model.track(
    source=0, 
    show=True, 
    save=True, 
    tracker="botsort.yaml" # BoT-SORT is a robust built-in tracking algorithm
)

In [ ]:
rf = Roboflow(api_key="mFPzVsqqKVbhmIKHlkb6")
project = rf.workspace("somework").project("chess-zn1zf")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
pwd

In [ ]:
?model.train

In [ ]:
from ultralytics import YOLO

# Load a pre-trained model to serve as our starting point
model = YOLO('yolo26n.pt')

# Train the model on your custom dataset
results = model.train(
    data='/Users/rafael/projects/ml_bootcamp/current/classwork/Chess-1/data.yaml', # Path to your dataset configuration
    epochs=50,                        # Number of times the model sees the entire dataset
    imgsz=640,                        # Standardize image size for training
    batch=64,                         # Number of images processed at once (lower this if your computer crashes)
    device='cpu'                     
)

print("Training complete! The new model is saved in the 'runs/detect/train/weights' folder.")

In [ ]:
from ultralytics import YOLO

# Create a new YOLO model from scratch
model = YOLO("yolo26n.yaml")

# Load a pretrained YOLO model (recommended for training)
model = YOLO("yolo26n.pt")

# Train the model using the 'coco8.yaml' dataset for 3 epochs
results = model.train(data="coco8.yaml", epochs=20)

# Evaluate the model's performance on the validation set
results = model.val()

# Perform object detection on an image using the model
results = model("https://ultralytics.com/images/bus.jpg")

# Export the model to ONNX format
success = model.export(format="onnx")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
model.export(format="onnx", dynamic=True)

In [ ]:
import cv2
from PIL import Image

from ultralytics import YOLO

model = YOLO("/Users/rafael/projects/ml_bootcamp/current/classwork/runs/detect/train-5/weights/best.pt")

results = model.predict(source="https://ultralytics.com/images/bus.jpg", save=True)




In [ ]:
results

Classwork


In [ ]:
rf = Roboflow(api_key="mFPzVsqqKVbhmIKHlkb6")

In [ ]:
import os
import csv
import shutil
from roboflow import Roboflow
from ultralytics import YOLO

# ==============================================================================
# STEP 1: DOWNLOAD THE DATASET
# ==============================================================================

from roboflow import Roboflow
rf = Roboflow(api_key="mFPzVsqqKVbhmIKHlkb6")
project = rf.workspace("somework").project("chess-zn1zf")
version = project.version(1)
dataset = version.download("yolo26")
                
dataset_root_dir = os.path.abspath(dataset.location)
print(f"Dataset downloaded to: {dataset_root_dir}")

# ==============================================================================
# STEP 2: AUTO-RESTRUCTURE MULTI-LABEL CSV INTO STANDARD CLASS FOLDERS
# ==============================================================================
# Roboflow classification exports usually include an '_annotations.csv' file mapping images to labels
splits = ['train', 'valid', 'test']

print("\n--- Restructuring dataset folders for YOLO Compatibility ---")
for split in splits:
    split_dir = os.path.join(dataset_root_dir, split)
    csv_path = os.path.join(split_dir, "_annotations.csv")
    
    if not os.path.exists(csv_path):
        # Fallback if Roboflow named it differently, list files to debug
        continue
        
    print(f"Processing {split} annotations...")
    
    with open(csv_path, mode='r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader) # Skip header row (typically: filename, label)
        
        for row in reader:
            if not row or len(row) < 2:
                continue
            filename = row[0].strip()
            # If multi-label strings contain commas (e.g. "king, queen"), take the primary/first one
            label = row[1].split(',')[0].strip() 
            
            # Define source path and new class folder path
            src_image_path = os.path.join(split_dir, filename)
            target_class_dir = os.path.join(split_dir, label)
            dst_image_path = os.path.join(target_class_dir, filename)
            
            # Create the class subfolder if it doesn't exist yet
            os.makedirs(target_class_dir, exist_ok=True)
            
            # Move the image inside its respective class folder
            if os.path.exists(src_image_path):
                shutil.move(src_image_path, dst_image_path)

    # Clean up the CSV file so YOLO doesn't get confused by its presence
    os.remove(csv_path)

print("Restructuring complete! All images have been sorted into class subfolders.")

# ==============================================================================
# STEP 3: INITIALIZE CLASSIFICATION MODEL AND TRAIN
# ==============================================================================
model = YOLO("yolo26n-cls.pt")

print("\n--- Initializing Classification Training Sequence ---")
training_results = model.train(
    data=dataset_root_dir,     # Now cleanly contains structured train/valid/test directories
    epochs=15,                 
    imgsz=224,                 
    batch=16,                  
    workers=4,                 
    device="cpu",              # Change to 0 for NVIDIA GPU or "mps" for Apple Silicon (Mac)
    project="Classroom_Chess", 
    name="yolo26_chess_cls"    
)

print("\n" + "="*80)
print("SUCCESS: Classification training completed.")
print("="*80)